# Spatiotemporal dynamics with scBIOT 1.2.0

This workflow combines spatial/time-aware integration, lineage-specific Schrödinger-bridge-style velocity fields, and transport-gene analysis. The input AnnData must contain a batch column, a time column, a lineage column, a representation in `obsm`, and spatial coordinates in `obsm['spatial']`.

## Setup

In [ ]:
import os
from pathlib import Path

import scanpy as sc
import scbiot as scb

tutorials_dir = Path(os.environ.get("SCBIOT_TUTORIALS_PATH", Path.cwd()))
adata = sc.read_h5ad(tutorials_dir / "inputs" / "spatiotemporal.h5ad")
adata

## Spatial/time-aware integration

When `time_key` is present, scBIOT preserves the ordered trajectory and disables automatic Gaussian prealignment.

In [ ]:
adata, metrics = scb.ot.integrate(
    adata,
    obsm_key="X_pca",
    batch_key="sample",
    out_key="X_scbiot_st",
    spatial_key="spatial",
    spatial_weight=0.5,
    time_key="timepoint",
    time_weight=0.5,
    time_mode="auto",
)
metrics

## Lineage-specific velocity field

In [ ]:
adata = scb.tl.velocity_field_sb_centroids(
    adata,
    obsm_key="X_scbiot_st",
    spatial_key="spatial",
    time_key="timepoint",
    lineage_key="lineage",
    out_vel_key="velocity_sb",
    time_bins=20,
    n_centroids_per_bin=512,
)

## Transport genes and energy

In [ ]:
ranked = scb.tl.rank_transport_score(
    adata,
    time_key="timepoint",
    lineage_key="lineage",
    rep_key="X_scbiot_st",
    store_key="transport_score",
    n_perms=200,
)

scb.tl.compute_transport_energy(
    adata, layer="transport_fwd", key_added="transport_energy", log1p=True
)

## Visualize

In [ ]:
sc.pp.neighbors(adata, use_rep="X_scbiot_st")
sc.tl.umap(adata)
scb.pl.umap_transport_energy(adata, key="transport_energy")
scb.pl.transport_gene_dynamics(
    adata, gene="SOX9", pseudotime_key="timepoint", layer="transport_fwd"
)